# ApexPlanet Internship - Task1

### ****Importing Libraries****

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('future.no_silent_downcasting', True)

### ****Import raw data****

In [2]:
df = pd.read_csv(r"C:\Users\Vinay\Downloads\archive (1).zip")

In [4]:
print("Initial Shape:", df.shape)


Initial Shape: (12575, 11)


In [5]:
print("\nColumn Info:")
print(df.info())




Column Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB
None


In [16]:
print(df.head())

  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    TXN_6867343     CUST_09     Patisserie   Item_10_PAT            18.5   
1    TXN_3731986     CUST_22  Milk Products  Item_17_MILK            29.0   
2    TXN_9303719     CUST_02       Butchers   Item_12_BUT            21.5   
3    TXN_9458126     CUST_06      Beverages   Item_16_BEV            27.5   
4    TXN_4575373     CUST_05           Food   Item_6_FOOD            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  Digital Wallet   Online       2024-04-08   
1       9.0        261.0  Digital Wallet   Online       2023-07-23   
2       2.0         43.0     Credit Card   Online       2022-10-05   
3       9.0        247.5     Credit Card   Online       2022-05-07   
4       7.0         87.5  Digital Wallet   Online       2022-10-02   

  Discount Applied  
0             True  
1             True  
2            False  
3              NaN  
4          

In [18]:
print(df.describe())

       Price Per Unit      Quantity   Total Spent
count    11966.000000  11971.000000  11971.000000
mean        23.365912      5.536380    129.652577
std         10.743519      2.857883     94.750697
min          5.000000      1.000000      5.000000
25%         14.000000      3.000000     51.000000
50%         23.000000      6.000000    108.500000
75%         33.500000      8.000000    192.000000
max         41.000000     10.000000    410.000000


## **Standardize Column Names**

In [6]:

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("\nCleaned Column Names:", df.columns)



Cleaned Column Names: Index(['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit',
       'quantity', 'total_spent', 'payment_method', 'location',
       'transaction_date', 'discount_applied'],
      dtype='object')


## ****Data Quality Assessment****

In [19]:
# Check for missing values in each column
print("Missing Values per Column:")
print(df.isnull().sum())

Missing Values per Column:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [20]:
# Check for duplicate rows
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")


Number of duplicate rows: 0


In [21]:
# Check data types (to identify inconsistent formatting)
print("\nData Types:")
print(df.dtypes)


Data Types:
Transaction ID       object
Customer ID          object
Category             object
Item                 object
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
Discount Applied     object
dtype: object


In [22]:
# Basic statistics to find outliers (e.g., negative prices or quantities)
df.describe()

,Price Per Unit,Quantity,Total Spent
count,11966.000000,11971.000000,11971.000000
mean,23.365912,5.536380,129.652577
std,10.743519,2.857883,94.750697
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,51.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


## ****Data Cleaning & Transformation****

In [7]:
# Handle Missing Values

print("\nMissing Values Before Cleaning:")
print(df.isnull().sum())


Missing Values Before Cleaning:
transaction_id         0
customer_id            0
category               0
item                1213
price_per_unit       609
quantity             604
total_spent          604
payment_method         0
location               0
transaction_date       0
discount_applied    4199
dtype: int64


In [8]:
# Numeric columns → fill with median
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [9]:
# Categorical columns → fill with "Unknown"
categorical_cols = df.select_dtypes(include="object").columns
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")


In [10]:
print("\nMissing Values After Cleaning:")
print(df.isnull().sum())



Missing Values After Cleaning:
transaction_id      0
customer_id         0
category            0
item                0
price_per_unit      0
quantity            0
total_spent         0
payment_method      0
location            0
transaction_date    0
discount_applied    0
dtype: int64


## **Trim Text & Standardize Format**

In [11]:
for col in categorical_cols:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.title()   # Standard case formatting


## **Standardize Date Columns**

In [12]:
# Automatically detect date columns
for col in df.columns:
    if "date" in col:
        df[col] = pd.to_datetime(df[col], errors="coerce")


In [13]:
# Create additional time features
for col in df.select_dtypes(include="datetime").columns:
    df[col + "_year"] = df[col].dt.year
    df[col + "_month"] = df[col].dt.month
    df[col + "_day"] = df[col].dt.day


## **Outlier Treatment**

In [14]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

## **Feature Engineering**

In [15]:
# Example: Customer Age from Date of Birth
if "date_of_birth" in df.columns:
    df["customer_age"] = (pd.Timestamp.now() - df["date_of_birth"]).dt.days // 365

# Example: Revenue Category
if "revenue" in df.columns:
    df["revenue_category"] = pd.cut(
        df["revenue"],
        bins=[0, 1000, 5000, 10000, df["revenue"].max()],
        labels=["Low", "Medium", "High", "Very High"]
    )

## **Final Data Validation**

In [16]:
print("\nFinal Shape:", df.shape)
print("\nFinal Data Types:")
print(df.dtypes)
print("\nFinal Missing Values:")
print(df.isnull().sum())


Final Shape: (12575, 14)

Final Data Types:
transaction_id                    object
customer_id                       object
category                          object
item                              object
price_per_unit                   float64
quantity                         float64
total_spent                      float64
payment_method                    object
location                          object
transaction_date          datetime64[ns]
discount_applied                  object
transaction_date_year              int32
transaction_date_month             int32
transaction_date_day               int32
dtype: object

Final Missing Values:
transaction_id               0
customer_id                  0
category                     0
item                         0
price_per_unit               0
quantity                     0
total_spent                  0
payment_method               0
location                     0
transaction_date             0
discount_applied          8376
tra

## **Export Cleaned Dataset**

In [17]:
df.to_csv("Retail_Sales_cleaned_dataset.csv", index=False)

print("\n✅ Cleaning Completed. File Saved as 'Retail_Sales_cleaned_dataset.csv'")


✅ Cleaning Completed. File Saved as 'Retail_Sales_cleaned_dataset.csv'
